# CECS 427 - Assignment 1

## Resources (Required Reading)

Research the NetworkX package:

* NetworkX Basics: https://networkx.org/documentation/stable/reference/introduction.html#networkx-basics

* NetworkX Tutorial: https://networkx.org/documentation/stable/tutorial.html

---

## Part A — NetworkX Practice with the Karate Club Graph

You will start with the Zachary Karate Club friendship network used in lecture.

### **Using NetworkX**

1. Load the Karate Club Graph

    * https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html

2. Draw the graph (using matplotlib)

3. Display basic network information:

    * number of nodes

    * number of edges

    * whether the graph is connected

    * average degree
    
    * density

4. Display:

    * Neighbors of node `0`

    * The distance (shortest path length) from node `0` to all other nodes

    * The shortest path from node `16` to node `23`



In [ ]:
# A1 & A2
# Imports
import networkx as nx
import matplotlib.pyplot as plt

"""
A1
"""
# Load graph from networkx
G = nx.karate_club_graph()

# Positioning
pos = nx.spring_layout(G)

"""
A2
"""
# Draw graph
nx.draw_networkx_nodes(G, pos)
nx.draw_networkx_edges(G, pos)
nx.draw_networkx_labels(G, pos)

# Show graph
plt.title("Zachary's Karate Club")
plt.show()

In [ ]:
# A3
# Imports
import networkx as nx

# Load graph from networkx
G = nx.karate_club_graph()

# Calculate average degree
degrees = dict(G.degree()) # Get all the nodes and degrees as a dict
avg_degree = sum(degrees.values()) / G.number_of_nodes() # Find the sum then divide by number of nodes for average

# Calcuate density
density = (2*G.number_of_edges()) / (G.number_of_nodes()*(G.number_of_nodes() - 1)) # 2E/(n(n-1)) for undirected graph

# Display information
print(f"Num nodes: {G.number_of_nodes()}")
print(f"Num edges: {G.number_of_edges()}")
print(f"Is connected?: {nx.is_connected(G)}")
print(f"Avg degree: {avg_degree}")
print(f"Density: {density}")

In [ ]:
# A4
# Imports
import networkx as nx

# Load graph from networkx
G = nx.karate_club_graph()

# Get neighbors of node 0
print(f"Neighbors of node 0: {list(G.neighbors(0))}")

# Shortest distance
print(f"Shortest paths from 0 (node: distance): {nx.shortest_path_length(G, source=0)}")

# Shortest distance from 16 to 23
print(f"Shortest path from from 16-23: {nx.shortest_path_length(G, source=16, target=23)}")


---

## Part B — Implement BFS from Scratch

*You must implement BFS without calling `nx.bfs_*` or `nx.shortest_path*` inside your BFS implementation.*

<br>

### **Implement BFS**

* Use a queue (e.g., `collections.deque`)

* `dist[source] = 0`

* For each reachable node `v`, `dist[v]` must be the shortest-path distance from source

* `paths[source]` should be `[source]`

* For each reachable node `v != source`, `paths[v]` should be a valid shortest path list

In [ ]:
from collections import deque


def bfs(G, source):
    """
    Returns:
      dist: dict[node] -> int
      paths: dict[node] -> list of nodes representing a shortest path from source to node
    """
    q = deque([source])
    dist = {source: 0}
    paths = {source: [source]}
    
    visited = {source: True}
    q.append(source)
    
    while q:
        node = q.popleft()
        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                visited[neighbor] = True
                dist[neighbor] = dist[node] + 1
                paths[neighbor] = paths[node] + [neighbor]
                q.append(neighbor)
    return dist, paths  
    

### **BFS Layers**

Implement a second function `bfs_layers(G, source)`:

* `layers[d]` is a sorted list

* `layers[0]` must contain only `[source]`

* Must be based on your BFS results (recommended: reuse `dist`)

In [ ]:
def bfs_layers(G, source):
    """
    Returns:
      layers: dict[int] -> sorted list of nodes at distance d from source
    """
    dist, paths = bfs(G, source)
    layers = {}
    for node, d in dist.items():
        if d not in layers:
            layers[d] = []
        layers[d].append(node)
    
    # Sort the nodes in each layer
    for d in layers:
        layers[d].sort()
    
    return layers


### **Test your BFS on Karate Graph**

* Run your BFS from source node `0` on the karate graph.

* Compare your BFS distances and shortest path against NetworkX. Print confirmation that they match.
    
    * The distance (shortest path length) from node 0 to all other nodes

    * The shortest path from node 16 to node 23

* Display BFS layers

In [ ]:
%pip install networkx

import networkx as nx
G = nx.karate_club_graph()

# Run BFS from node 0
dist, paths = bfs(G, 0)

print("BFS distances from node 0:")
print(dist)

print("\nBFS shortest path from node 0 to every reachable node:")
for node in sorted(paths):
    print(f"0 -> {node}: {paths[node]}")

# BFS layers
layers = bfs_layers(G, 0)

print("\nBFS layers:")
for distance, nodes in layers.items():
    print(f"Layer {distance}: {nodes}")

# Compare BFS distances with NetworkX
nx_dist = dict(nx.single_source_shortest_path_length(G, 0))

print("\nDistance comparison:")
print(dist == nx_dist)

# Compare the path from node 16 to node 23
bfs_path = paths[23]  # This is the path from source 0 to node 23

nx_path = nx.shortest_path(G, source=0, target=23)

print("\nBFS path from 0 to 23:")
print(bfs_path)

print("\nNetworkX path from 0 to 23:")
print(nx_path)

# Compare the requested path from node 16 to node 23
bfs_dist_16, bfs_paths_16 = bfs(G, 16)
bfs_path_16_to_23 = bfs_paths_16[23]

nx_path_16_to_23 = nx.shortest_path(G, source=16, target=23)

print("\nBFS shortest path from 16 to 23:")
print(bfs_path_16_to_23)

print("\nNetworkX shortest path from 16 to 23:")
print(nx_path_16_to_23)

print("\nPath lengths match:")
print(len(bfs_path_16_to_23) - 1 == len(nx_path_16_to_23) - 1)

---

## Part C — Erdős–Rényi Random Graphs

Implement and analyze Erdős–Rényi random graphs using only the definition and formula introduced in lecture.

* You may use NetworkX to create an empty graph and add edges.

* Do not use `nx.gnp_random_graph()` or other NetworkX random-graph generators.

<br>

### **Generate Random Graphs with `c = 0.4`, `c = 0.6`, and `c = 1.5`**

* Let `n` be number of nodes in the karate graph

* For each given constant `c`, compute `p`

Generate **three** random graphs for each value of `c`:

1. For each graph, display:
    
    * number of nodes

    * number of edges
    
    * density

    * whether the graph is connected?
    
    * whether the graph contains a 'giant component' (fraction >= 0.5)?

    * number of connected components

    * list of component sizes (descending)

    * largest connected component size

    * largest component fraction

2. Draw the graphs
    * Include the value of `c` (and/or `p`) in the figure title


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import math

karate = nx.karate_club_graph()
n = karate.number_of_nodes()

for c in [0.4, 0.6, 1.5]:
    p = c * math.log(n) / n

    for trial in range(1, 4):
        G = nx.Graph()
        G.add_nodes_from(range(n))
        for u in range(n):
            for v in range(u + 1, n):
                if random.random() < p:
                    G.add_edge(u, v)

        components = []
        for comp in nx.connected_components(G):
            components.append(len(comp))
        components.sort()

        largestFraction = components[-1] / n

        print(f"c = {c}, p = {p:.4f}, trial {trial}")
        print(f"Num nodes: {G.number_of_nodes()}")
        print(f"Num edges: {G.number_of_edges()}")
        print(f"Density: {nx.density(G)}")
        print(f"Is connected?: {nx.is_connected(G)}")
        print(f"Has giant component?: {largestFraction >= 0.5}")
        print(f"Number of components: {len(components)}")
        print(f"Component sizes: {components}")
        print(f"Largest component size: {components[-1]}")
        print(f"Largest component fraction: {largestFraction}")
        print()

        pos = nx.spring_layout(G)
        plt.figure(figsize=(5, 5))
        nx.draw(G, pos, with_labels=True)
        plt.title(f"Erdos Renyi Graph (c={c}, p={p:.3f}), trial {trial}")
        plt.show()

### **Conceptual Questions** (Answer in Comments or Markdown)

1. For which values of `c` does a giant component appear?

2. How does increasing `c` affect connectivity and component sizes?

3. How do your observations compare with the Giant Component Theorem discussed in lecture?

1) The values of C where a giant component will appear is when c > 1 because the Giant component theorem states that it will appear with high probability at P>= ln(n)/n.
2) Increasing C affects the connectivity and component sizes by adding more edges which reduces the amount of components that are separated and increases the largest one. Different components will also merge together.
3) Our observations compare with the theorem dicussed in lecture by matching the prediction and showing that giant components will appear most of the time when above 1 and not all the time when c is less than 1.